# 🕵️ Dia 62 — Lendo relatórios de sandbox sem se enganar

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 62: Reading sandbox reports without fooling yourself* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-62.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O técnico da Serra Azul rodou o executável numa *sandbox* gratuita: *"Deu 10/10 malicioso!! E apareceu 'Credential Access'."* Rafa traçou uma linha no quadro: de um lado **observado**, do outro **pontuado**. *"O relatório para a Marta só usa a coluna da esquerda."*

Neste notebook você vai:

1. **Carregar dois relatórios sintéticos** (instalador do 7-Zip e `msa-atualizador.exe`), inventados para o curso.
2. **Rodar o fluxograma do Conceito como código**: observado/pontuado, amostra/ambiente, esperado/inesperado.
3. **Avaliar os rótulos ATT&CK** procurando o comportamento que disparou cada um.
4. **Gerar a planilha, os planos de *sandbox* privada** e o parágrafo para a Marta.
5. **Jogar o quiz "isso entra no dossiê?"**.

> ⚠️ Nenhum arquivo é executado ou enviado. Os relatórios abaixo são **sintéticos** (`"synthetic": true`), num formato simplificado que não copia nenhum serviço real. Para o exercício real do original, use os relatórios públicos do instalador do 7-Zip que você obtiver.

## 1. Os dois relatórios (sintéticos)

In [ ]:
# Formato simplificado e inventado. Cada evento tem: tipo, processo (pid/nome) e detalhe.
# O PID 1 é sempre a amostra; os filhos dela herdam 'arvore'. Processos fora da árvore = ambiente.
REL_7ZIP = {
    "synthetic": True, "sandbox": "Sandbox A (exemplo)", "url": "https://sandbox-a.example/report/7z-0001",
    "data": "2026-10-07", "imagem": "Windows 10 22H2 x64", "duracao_s": 240, "interacao": "cliques no instalador",
    "veredito": "No specific threat", "nota": "0/100",
    "processos": [
        {"pid": 1, "ppid": 0, "nome": "7z2408-x64.exe", "cmd": "7z2408-x64.exe"},
        {"pid": 2, "ppid": 1, "nome": "regsvr32.exe", "cmd": "regsvr32 /s \"C:\\Program Files\\7-Zip\\7-zip.dll\""},
        {"pid": 90, "ppid": 0, "nome": "svchost.exe", "cmd": "svchost.exe -k netsvcs"},
    ],
    "eventos": [
        {"tipo": "arquivo", "pid": 1, "detalhe": "C:\\Program Files\\7-Zip\\7z.exe"},
        {"tipo": "arquivo", "pid": 1, "detalhe": "C:\\Program Files\\7-Zip\\7-zip.dll"},
        {"tipo": "arquivo", "pid": 1, "detalhe": "C:\\Program Files\\7-Zip\\7zFM.exe"},
        {"tipo": "registro", "pid": 2, "detalhe": "HKLM\\SOFTWARE\\Classes\\CLSID\\{23170F69-40C1-278A-1000-000100020000}"},
        {"tipo": "registro", "pid": 1, "detalhe": "HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Uninstall\\7-Zip"},
        {"tipo": "rede", "pid": 90, "detalhe": "TCP 192.0.2.50:443 (Windows Update)"},
    ],
    "attck": [
        {"id": "T1112", "nome": "Modify Registry", "gatilho": "registro"},
        {"id": "T1218.010", "nome": "System Binary Proxy Execution: Regsvr32", "gatilho": "processo:regsvr32.exe"},
    ],
    "esperado": ["Program Files\\7-Zip", "Classes", "Uninstall\\7-Zip", "regsvr32"],
}

REL_MSA = {
    "synthetic": True, "sandbox": "Sandbox privada da Vigia (fictícia)", "url": "interno://vigia/sandbox/ISCA-0007",
    "data": "2026-10-08", "imagem": "Windows 10 22H2 x64 pt-BR", "duracao_s": 300,
    "interacao": "usuário simulado clicou 'Avançar' e 'Concluir'",
    "veredito": "Malicious", "nota": "10/10",
    "processos": [
        {"pid": 1, "ppid": 0, "nome": "msa-atualizador.exe", "cmd": "msa-atualizador.exe"},
        {"pid": 2, "ppid": 1, "nome": "schtasks.exe", "cmd": "schtasks /create /tn \"MSA Atualizacao Automatica\" /sc minute /mo 5"},
        {"pid": 3, "ppid": 1, "nome": "cmd.exe", "cmd": "cmd /c systeminfo"},
        {"pid": 4, "ppid": 3, "nome": "systeminfo.exe", "cmd": "systeminfo"},
        {"pid": 90, "ppid": 0, "nome": "svchost.exe", "cmd": "svchost.exe -k netsvcs"},
        {"pid": 91, "ppid": 0, "nome": "svchost.exe", "cmd": "svchost.exe -k LocalService (CryptSvc)"},
    ],
    "eventos": [
        {"tipo": "arquivo", "pid": 1, "detalhe": "C:\\Users\\analista\\AppData\\Roaming\\MSA\\config.ini"},
        {"tipo": "arquivo", "pid": 1, "detalhe": "C:\\Users\\analista\\AppData\\Local\\Temp\\msa_0001.tmp"},
        {"tipo": "dns", "pid": 1, "detalhe": "atualiza-modulo.example → 198.51.100.140"},
        {"tipo": "rede", "pid": 1, "detalhe": "TCP 198.51.100.140:443, 1.312 bytes enviados"},
        {"tipo": "rede", "pid": 1, "detalhe": "TCP 198.51.100.140:443, 1.298 bytes enviados (300 s depois)"},
        {"tipo": "string", "pid": 1, "detalhe": "Login Data"},
        {"tipo": "rede", "pid": 90, "detalhe": "TCP 192.0.2.50:443 (Windows Update)"},
        {"tipo": "rede", "pid": 91, "detalhe": "HTTP 192.0.2.60:80 /ocsp (revogação de certificado)"},
    ],
    "attck": [
        {"id": "T1053.005", "nome": "Scheduled Task/Job: Scheduled Task", "gatilho": "processo:schtasks.exe"},
        {"id": "T1082", "nome": "System Information Discovery", "gatilho": "processo:systeminfo.exe"},
        {"id": "T1071.001", "nome": "Application Layer Protocol: Web Protocols", "gatilho": "rede"},
        {"id": "T1041", "nome": "Exfiltration Over C2 Channel", "gatilho": "rede",
         "ressalva": "a conexão foi observada; 'exfiltração' é interpretação da regra: só ~1,3 KB por conexão, conteúdo não visto"},
        {"id": "T1555.003", "nome": "Credentials from Password Stores: Credentials from Web Browsers", "gatilho": "string"},
    ],
    "esperado": [],     # um "módulo de segurança" da cooperativa não deveria existir: a cooperativa não distribui nenhum
}
print("Relatórios sintéticos carregados:", REL_7ZIP["sandbox"], "|", REL_MSA["sandbox"])

## 2. O fluxograma do Conceito, como código

Para cada evento: é da árvore da amostra? É esperado para o que o arquivo diz ser? Vira indicador?

In [ ]:
def arvore_da_amostra(rel):
    """PIDs descendentes do PID 1 (a amostra)."""
    filhos = {1}
    mudou = True
    while mudou:                                              # repete até não achar mais filhos
        mudou = False
        for p in rel["processos"]:
            if p["ppid"] in filhos and p["pid"] not in filhos:
                filhos.add(p["pid"]); mudou = True
    return filhos

def classificar(rel):
    arv = arvore_da_amostra(rel)
    nomes = {p["pid"]: p["nome"] for p in rel["processos"]}
    linhas = []
    for e in rel["eventos"]:
        origem = "amostra" if e["pid"] in arv else "ambiente"
        if e["tipo"] == "string":
            natureza = "string casada por regra (não é ação)"
        else:
            natureza = "observado"
        esperado = any(x in e["detalhe"] for x in rel["esperado"])
        if origem == "ambiente":
            destino = "ruído: rotule e deixe de fora"
        elif e["tipo"] == "string":
            destino = "contexto (não é comportamento)"
        elif esperado:
            destino = "consistente com a finalidade declarada"
        else:
            destino = f"INDICADOR (cite {rel['url']}, {rel['data']})"
        linhas.append((e["tipo"], nomes[e["pid"]], e["detalhe"], natureza, origem, destino))
    return linhas

for rel in (REL_7ZIP, REL_MSA):
    print(f"\n=== {rel['sandbox']} — veredito '{rel['veredito']}' / nota {rel['nota']} → CONTEXTO, não achado")
    for tipo, proc, det, nat, orig, dest in classificar(rel):
        print(f"[{tipo:8s}] {proc:20s} {orig:8s} → {dest}\n            {det}")

## 3. Avaliando os rótulos ATT&CK

In [ ]:
def avaliar_attck(rel):
    arv = arvore_da_amostra(rel)
    nomes = {p["pid"]: p["nome"] for p in rel["processos"]}
    saida = []
    for t in rel["attck"]:
        gat = t["gatilho"]
        if gat.startswith("processo:"):                          # rótulo disparado por um processo filho
            ev = [p["cmd"] for p in rel["processos"] if p["nome"] == gat.split(":")[1] and p["pid"] in arv]
        else:                                                    # disparado por um tipo de evento da amostra
            ev = [e["detalhe"] for e in rel["eventos"] if e["tipo"] == gat and e["pid"] in arv]
        if not ev:
            veredito = "sem comportamento por trás: regra casou, NÃO observado"
        elif gat == "string":
            veredito = "regra casou numa STRING; nenhuma ação correspondente observada"
        elif t.get("ressalva"):
            veredito = "PARCIAL: " + t["ressalva"]
        else:
            veredito = "descrição exata do comportamento observado"
        saida.append((t["id"], t["nome"], ev[0] if ev else "—", veredito))
    return saida

for rel in (REL_7ZIP, REL_MSA):
    print(f"\n=== {rel['sandbox']}")
    for tid, nome, ev, v in avaliar_attck(rel):
        print(f"{tid:10s} {nome[:45]:45s}\n           gatilho: {ev[:70]}\n           → {v}")
print("""
No 7-Zip: T1112 e T1218.010 são descrições exatas e NÃO maliciosas no contexto (instalador registra
a própria DLL e a entrada de desinstalação). No MSA: T1555.003 vem só de uma string ('Login Data'),
e T1041 nomeia como 'exfiltração' conexões de ~1,3 KB cujo conteúdo ninguém viu.""")

### 🧩 Exercício

Qual rótulo do relatório do MSA você levaria para o informe como **comportamento observado**, e qual entraria só como "o relatório rotula, mas não registra"? Responda na próxima célula de quiz e confira.

## 4. Planilha, planos privados e o parágrafo para a Marta

In [ ]:
def planilha(rel_a, rel_b=None):
    def campo(rel, tipo):
        if rel is None:
            return "—"
        arv = arvore_da_amostra(rel)
        itens = [e["detalhe"] for e in rel["eventos"] if e["tipo"] == tipo and e["pid"] in arv][:5]
        return "<br>".join(itens) if itens else "não mostrado por esta sandbox"
    linhas = ["| Campo | Sandbox A | Sandbox B |", "|---|---|---|"]
    for nome, f in [("Veredito / nota (CONTEXTO)", lambda r: f"{r['veredito']} {r['nota']}" if r else "—"),
                    ("Árvore de processos", lambda r: " → ".join(p["nome"] for p in r["processos"] if p["pid"] in arvore_da_amostra(r)) if r else "—"),
                    ("Arquivos gravados", lambda r: campo(r, "arquivo")),
                    ("Registro", lambda r: campo(r, "registro")),
                    ("DNS", lambda r: campo(r, "dns")),
                    ("Rede (amostra)", lambda r: campo(r, "rede")),
                    ("ATT&CK", lambda r: ", ".join(t["id"] for t in r["attck"]) if r else "—")]:
        linhas.append(f"| {nome} | {f(rel_a)} | {f(rel_b)} |")
    return "\n".join(linhas)

print(planilha(REL_7ZIP))   # a Sandbox B você preenche com o segundo relatório real

In [ ]:
#@title Plano de sandbox privada { display-mode: "form" }
tempo_execucao_min = 15  #@param {type:"integer"}
interacao = "clicar Avançar/Concluir; abrir o navegador; esperar 10 min ociosa"  #@param {type:"string"}

PLANOS = f"""# Planos de sandbox privada

## synchelper.exe (LAB-P4)
- Onde: sandbox privada (CAPE auto-hospedado ou camada paga privada). Nunca pública.
- Imagem: Windows na mesma build da WS-FIN-07, com Office/Outlook (o pai foi powershell.exe filho do OUTLOOK.EXE).
- Interação: {interacao}. Tempo: {tempo_execucao_min} min.
- Procurar: conexões para 198.51.100.23; filho rundll32.exe sem argumentos; gravações em AppData\\Roaming\\SyncHelper.
- Por que não pública: arquivo da estação de um cliente pode conter dados dele e o envio pode alertar quem o escreveu.

## ➕ msa-atualizador.exe (Isca)
- Onde: sandbox privada da Vigia (feito em 2026-10-08, relatório interno ISCA-0007).
- Imagem: Windows 10 22H2 pt-BR (localidade do alvo; amostras podem checar idioma).
- Procurar: atualiza-modulo.example / 198.51.100.140, tarefa "MSA Atualizacao Automatica", leitura de Documents (limite_mb=3 no config.ini).
- Por que não pública: estação de cooperativa de crédito (sigilo bancário, LGPD) e kit possivelmente direcionado à Serra Azul.
"""
PARAGRAFO = """Nesta execução, neste ambiente (Windows 10 pt-BR, 300 s, sandbox privada, 2026-10-08), o arquivo msa-atualizador.exe
criou uma tarefa agendada chamada "MSA Atualizacao Automatica" para rodar a cada 5 minutos e executou o comando systeminfo.
Ele consultou o nome atualiza-modulo.example e abriu duas conexões HTTPS para 198.51.100.140, com cerca de 1,3 KB enviados em cada,
com 300 s de intervalo, o mesmo ritmo visto na rede da cooperativa. Não foi observada leitura de senhas do navegador nem envio de
documentos durante a execução; isso não exclui que ocorram em outras condições. A cooperativa não distribui nenhum "módulo de
segurança", portanto a presença do arquivo nas estações deve ser tratada como incidente."""
from pathlib import Path
Path("notas").mkdir(exist_ok=True)
Path("notas/planos-sandbox.md").write_text(PLANOS, encoding="utf-8")
Path("notas/paragrafo-marta.md").write_text(PARAGRAFO + "\n", encoding="utf-8")
print(PARAGRAFO)
proibidas = [p for p in ("10/10", "malicioso", "Malicious", "Trojan") if p in PARAGRAFO]
print("\nPalavras de pontuação no parágrafo:", proibidas or "nenhuma ✅")

## 5. 🎮 Quiz: isso entra no dossiê?

Checkpoint BR: pelo menos 5 de 6.

In [ ]:
# ------------------------------------------------------------
# QUIZ: função genérica de múltipla escolha
# MODO_JOGO = True  -> pergunta e conta acertos
# MODO_JOGO = False -> só mostra gabarito e explicação
# ------------------------------------------------------------
def jogar_quiz(perguntas, modo_jogo=True, minimo=None):
    acertos = 0                                            # placar
    for n, (enunciado, opcoes, certa, porque) in enumerate(perguntas, start=1):
        print(f"\n[{n}/{len(perguntas)}] {enunciado}")    # o enunciado
        for letra, texto in opcoes.items():                # as alternativas a), b), c)
            print(f"   {letra}) {texto}")
        if modo_jogo:
            resposta = ""
            for _ in range(5):                             # até 5 tentativas de digitar uma letra válida
                resposta = input("   Sua resposta: ").strip().lower()
                if resposta in opcoes:
                    break
            ok = (resposta == certa)                       # acertou?
            acertos += ok
            print("   ✅ Acertou!" if ok else f"   ❌ Era {certa}).")
        else:
            print(f"   Gabarito: {certa})")
        print(f"   Por quê: {porque}")
    if modo_jogo:
        print(f"\n🏁 Resultado: {acertos}/{len(perguntas)}")
        if minimo is not None:
            print("Checkpoint BR cumprido. 🥷" if acertos >= minimo
                  else f"Checkpoint BR pede pelo menos {minimo}. Releia o README e tente de novo.")
    return acertos

In [ ]:
MODO_JOGO = True
PERGUNTAS = [
    ("'Veredito: Malicious, 10/10'. No dossiê entra como:",
     {"a": "achado principal", "b": "contexto, numa linha, no fim", "c": "não entra"},
     "b", "A nota é opinião de regras; o achado são as observações."),
    ("Conexão de svchost.exe (netsvcs) para Windows Update durante a execução:",
     {"a": "indicador do caso", "b": "ruído do ambiente: rotule e deixe de fora", "c": "exfiltração"},
     "b", "Não está na árvore de processos da amostra."),
    ("T1112 'Modify Registry' na instalação do 7-Zip:",
     {"a": "acusação de persistência", "b": "descrição exata, não maliciosa no contexto (entrada de desinstalação)", "c": "falso positivo a apagar"},
     "b", "É o item 'sem notas' do checkpoint do original."),
    ("T1555.003 no relatório do MSA, disparado só pela string 'Login Data':",
     {"a": "roubo de senhas confirmado", "b": "regra casou numa string; comportamento não observado", "c": "irrelevante"},
     "b", "String sugere capacidade; ação não foi registrada nesta execução."),
    ("Relatório quase silencioso, nota 0/100. Você escreve:",
     {"a": "benigno", "b": "nada observável neste ambiente, neste tempo", "c": "inconclusivo, descartar"},
     "b", "Mais fraco do que benigno: VM detectada, espera, falta de interação."),
    ("O técnico já subiu o arquivo numa sandbox pública. O que fazer?",
     {"a": "nada, já foi", "b": "registrar o envio (serviço, data, visibilidade), pedir remoção se possível e informar o encarregado",
      "c": "apagar o print para ninguém saber"},
     "b", "É um incidente dentro do incidente; transparência e registro."),
]
jogar_quiz(PERGUNTAS, MODO_JOGO, minimo=5)

In [ ]:
try:
    from google.colab import files
    for f in ("notas/planos-sandbox.md", "notas/paragrafo-marta.md"):
        files.download(f)
except ImportError:
    print("(Fora do Colab: os arquivos ficaram em notas/.)")

## ✅ Antes de fechar o Dia 62

- [ ] Planilha das duas *sandboxes* (relatórios reais do 7-Zip) com URLs e datas
- [ ] Ao menos um rótulo ATT&CK avaliado como "descrição exata, não malicioso no contexto"
- [ ] Planos privados do `synchelper.exe` e do `msa-atualizador.exe`
- [ ] Parágrafo para a Marta sem nota e sem nome de família
- [ ] Quiz com pelo menos 5/6

**Amanhã (Dia 63):** SIEM. Uma consulta só vai mostrar todas as estações que falaram com 198.51.100.140, e o que aconteceu com a conta da caixa.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>